# 04 · ข้อ 3 ผู้โดยสารระหว่างประเทศ ขาเข้า vs ขาออก
ผู้รับผิดชอบ: สมาชิก C · ต้องรัน `01_cleaning.ipynb` ก่อน

**คำถาม:** จำนวนผู้โดยสารระหว่างประเทศขาเข้าและขาออกมีความแตกต่างกันอย่างไร เมื่อจำแนกตามรูปแบบการเดินทาง เปรียบเทียบรายเดือน และวิเคราะห์ความผันผวนภายในปี

โครงของ notebook ตรงกับหัวข้อที่รายงานต้องมีในแต่ละข้อ: Feature Selection → Data Preparation → Feature Engineering → Visualization → Analysis & Findings

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import prep

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
prep.setup_thai_font()

'Loma'

In [2]:
df = prep.load_clean()
catalog = pd.read_csv(prep.PREPARED_DIR / "service_catalog.csv", encoding="utf-8-sig", keep_default_na=False)
print(df.shape, df["year_be"].value_counts().to_dict())

(18705, 18) {2568: 15695, 2569: 3010}


## 1. Feature Selection

| Feature ต้นฉบับ | เหตุผล |
|---|---|
| `purpose` | เลือกเฉพาะ international |
| `mode` | จำแนกตามรูปแบบการเดินทาง (ถนน/น้ำ/ราง/อากาศ) |
| `port`, `direction` | จับคู่ขาเข้า–ขาออกของท่าเดียวกัน |
| `date`, `value` | รวมเป็นรายเดือน |

## 2. Data Preparation

In [3]:
# ระหว่างประเทศ · หน่วย "คน" · 7 คู่ขาเข้า/ขาออก
q3 = prep.passenger_rows(df)
q3 = q3[q3["purpose"] == "international"].copy()
pairs = q3.groupby(["mode", "port"])["direction"].unique()
display(pairs)
assert all(set(d) == {"arrival", "departure"} for d in pairs), "every port must have both directions"

# ความครบถ้วนรายเดือน (ก่อนรวมเป็นรายเดือนต้องรู้ว่าแต่ละเดือนมีวันที่ใช้ได้ครบหรือไม่)
coverage = (q3.assign(month=q3["date"].dt.to_period("M"))
              .groupby(["year_be", "mode", "port", "direction", "month"])
              .agg(days=("value", "size"), valid=("value", "count")))
coverage[coverage["valid"] < coverage["days"]]

mode   port                      
air    ท่าอากาศยานดอนเมือง           [departure, arrival]
       ท่าอากาศยานภูมิภาค            [departure, arrival]
       ท่าอากาศยานสุวรรณภูมิ         [departure, arrival]
       ท่าอากาศยานอื่น ๆ ของ ทอท.    [departure, arrival]
rail   รถไฟ                          [departure, arrival]
road   รถ บขส.                       [departure, arrival]
water  ท่าเรือด่านชายแดน             [departure, arrival]
Name: direction, dtype: object

days  valid
year_be mode port    direction month               
2568    road รถ บขส. arrival   2025-02    28     27
                               2025-03    31     30
                               2025-04    30     29
                               2025-10    31     30
                               2025-11    30     29
                               2025-12    31     30
                     departure 2025-02    28     27
                               2025-03    31     30
                               2025-04    30     29
                               2025-06    30     29
                               2025-10    31     29
                               2025-11    30     29
                               2025-12    31     29
2569    road รถ บขส. arrival   2026-02    28     25
                     departure 2026-02    28     25

## 3. Feature Engineering (อย่างน้อย 3 ตัวแปร)

| Feature ใหม่ | สูตร / เงื่อนไข | ประโยชน์ |
|---|---|---|
| `monthly_total` | sum ต่อเดือน × mode × direction (ถ้าเดือนไม่ครบ ใช้ค่าเฉลี่ยต่อวัน × จำนวนวัน หรือรายงานเป็นค่าเฉลี่ยต่อวัน) | ระดับรายเดือน |
| `net_flow` | `arrival − departure` | ทิศทางสุทธิ |
| `arr_dep_ratio` | `arrival / departure` | เทียบ mode ที่ขนาดต่างกันมากได้ |
| `mom_pct_change` | `(m − m₋₁) / m₋₁ × 100` | เดือนที่เปลี่ยนมากที่สุด |
| `cv_within_year` | `std(monthly_total) / mean(monthly_total)` ต่อ mode × direction | ความผันผวนภายในปีเป็นตัวเลขเดียว |
| `mode_share` | สัดส่วนของ mode ต่อยอดระหว่างประเทศของเดือน | mode ใดเป็นหลัก |

หมายเหตุ: ทางอากาศใหญ่กว่า mode อื่นหลายสิบเท่า → ใช้ ratio/share หรือแยก panel ไม่ใช่แกนเดียว

In [4]:
# TODO: สร้าง features ตามตารางด้านบน แล้วแสดง .head() และ .describe() เป็นหลักฐาน

## 4. Visualization (อย่างน้อย 2 รูปแบบ)
แนะนำ: ① line รายเดือน ขาเข้า vs ขาออก แยก panel ตาม mode ② bar ของ `net_flow` หรือ `arr_dep_ratio` รายเดือน ③ bar ของ `cv_within_year`

บันทึกทุกกราฟด้วย `prep.save_fig(fig, "<ชื่อ>")` แล้วใช้ไฟล์ใน `figures/` ในรายงานและสไลด์

In [5]:
# TODO: กราฟที่ 1

**การตีความกราฟที่ 1:** _(เขียนจากตัวเลขจริงในกราฟ)_

In [6]:
# TODO: กราฟที่ 2

**การตีความกราฟที่ 2:** _(เขียนจากตัวเลขจริงในกราฟ)_

## 5. Analysis & Findings
- คำตอบของคำถาม:
- ข้อค้นพบสำคัญ:
- ข้อจำกัดของข้อมูล:

## 6. บันทึก Prepared Dataset

In [7]:
# TODO: เปลี่ยน result_df เป็นตารางสุดท้ายที่มี features ใหม่ครบ
# result_df.to_csv(prep.PREPARED_DIR / "q3_international_monthly.csv", index=False, encoding="utf-8-sig")